# Generation Expansion DBC/iDBC Notebook

Use this notebook to run minimal generation-expansion experiments with either `DBC` or `iDBC`.


In [ ]:
using Pkg

function find_repo_root(start::AbstractString = pwd())
    dir = abspath(start)
    while true
        has_project = isfile(joinpath(dir, "Project.toml"))
        has_entry = isfile(joinpath(dir, "experiments", "generation_expansion", "run_experiment.jl"))
        if has_project && has_entry
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repository root from $(start).")
        dir = parent
    end
end

repo_root = find_repo_root()
cd(repo_root)
Pkg.activate(repo_root)
println("repo_root = ", repo_root)


## Edit Parameters

`cut_mode` can be `:DBC` or `:iDBC`.


In [ ]:
run_script = joinpath("experiments", "generation_expansion", "run_experiment.jl")
config_path = joinpath("notebooks", "tmp_ge_notebook_config.jl")

cut_mode = :iDBC                     # :DBC or :iDBC
enable_copy_branching = (cut_mode == :iDBC)
enable_surrogate_copy_split = true
idbc_warm_pass = false

periods = 6
realizations = 6

num_workers = 5
dry_run = false
max_iter = 5
sample_count = 20
num_backward_samples = 1
terminate_time = 300.0
time_limit = 20.0
experiment_tag = "nb_ge_$(cut_mode)"

config_text = """
EXPERIMENT_CONFIG = GeExperimentConfig(
    num_workers = $(num_workers),
    dry_run = $(dry_run),
    static = GeStaticConfig(
        max_iter = $(max_iter),
        sample_count = $(sample_count),
        num_backward_samples = $(num_backward_samples),
        terminate_time = $(terminate_time),
        time_limit = $(time_limit),
        enable_copy_branching = $(enable_copy_branching),
        enable_surrogate_copy_split = $(enable_surrogate_copy_split),
        copy_split_strategy = :surrogate_delta,
        idbc_warm_pass = $(idbc_warm_pass),
        experiment_tag = $(repr(experiment_tag)),
        legacy_logger_paths = false,
    ),
    sweep = GeSweepConfig(
        cuts = Symbol[$(repr(cut_mode))],
        realizations = Int[$(realizations)],
        periods = Int[$(periods)],
    ),
)
"""

write(joinpath(repo_root, config_path), config_text)
println("Wrote config: ", joinpath(repo_root, config_path))
println(config_text)


In [ ]:
cd(repo_root) do
    run(`julia --project=. $run_script $config_path`)
end
